# Unit 4.1 Naïve Bayes Exercise

This notebook implements Naïve Bayes in two ways using the provided HAM/SPAM dataset:
1. **Manual implementation** (Bag of Words, priors, likelihoods, log-posterior prediction)
2. **Scikit-learn implementation** (`CountVectorizer` + `MultinomialNB`)

Required test sentences:
- `Limited offer, click here!`
- `Meeting at 2 PM with the manager.`

## 1) Define Dataset and Text Preprocessing

Create the dataset and a reusable preprocessing function:
- lowercase
- remove punctuation
- tokenize by whitespace

In [1]:
import re
import math
from collections import Counter, defaultdict

# Training data from the activity sheet
train_docs = [
    "Free money now!!!",
    "Hi mom, how are you?",
    "Lowest price for your meds",
    "Are we still on for dinner?",
    "Win a free iPhone today",
    "Let's catch up tomorrow at the office",
    "Meeting at 3 PM tomorrow",
    "Get 50% off, limited time!",
    "Team meeting in the office",
    "Click here for prizes!",
    "Can you send the report?",
]

train_labels = [
    "SPAM", "HAM", "SPAM", "HAM", "SPAM",
    "HAM", "HAM", "SPAM", "HAM", "SPAM", "HAM"
]

def preprocess(text: str):
    """Lowercase, remove punctuation, then split into tokens."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return tokens

tokenized_docs = [preprocess(doc) for doc in train_docs]
for i, (doc, label, tokens) in enumerate(zip(train_docs, train_labels, tokenized_docs), start=1):
    print(f"{i:>2}. [{label}] {doc}")
    print(f"    tokens: {tokens}")

 1. [SPAM] Free money now!!!
    tokens: ['free', 'money', 'now']
 2. [HAM] Hi mom, how are you?
    tokens: ['hi', 'mom', 'how', 'are', 'you']
 3. [SPAM] Lowest price for your meds
    tokens: ['lowest', 'price', 'for', 'your', 'meds']
 4. [HAM] Are we still on for dinner?
    tokens: ['are', 'we', 'still', 'on', 'for', 'dinner']
 5. [SPAM] Win a free iPhone today
    tokens: ['win', 'a', 'free', 'iphone', 'today']
 6. [HAM] Let's catch up tomorrow at the office
    tokens: ['let', 's', 'catch', 'up', 'tomorrow', 'at', 'the', 'office']
 7. [HAM] Meeting at 3 PM tomorrow
    tokens: ['meeting', 'at', '3', 'pm', 'tomorrow']
 8. [SPAM] Get 50% off, limited time!
    tokens: ['get', '50', 'off', 'limited', 'time']
 9. [HAM] Team meeting in the office
    tokens: ['team', 'meeting', 'in', 'the', 'office']
10. [SPAM] Click here for prizes!
    tokens: ['click', 'here', 'for', 'prizes']
11. [HAM] Can you send the report?
    tokens: ['can', 'you', 'send', 'the', 'report']


## 2) Generate Bag-of-Words Vocabulary and Frequency Tables

Build full vocabulary and class-wise token frequencies.

In [2]:
vocab = sorted(set(token for doc in tokenized_docs for token in doc))
V = len(vocab)

class_token_counts = {
    "HAM": Counter(),
    "SPAM": Counter(),
}
class_total_tokens = {
    "HAM": 0,
    "SPAM": 0,
}

for tokens, label in zip(tokenized_docs, train_labels):
    class_token_counts[label].update(tokens)
    class_total_tokens[label] += len(tokens)

print(f"Vocabulary size: {V}")
print("Vocabulary:", vocab)
print("\nHAM token counts:")
print(dict(class_token_counts["HAM"]))
print("\nSPAM token counts:")
print(dict(class_token_counts["SPAM"]))
print("\nTotal tokens per class:")
print(class_total_tokens)

Vocabulary size: 45
Vocabulary: ['3', '50', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', 'let', 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 's', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']

HAM token counts:
{'hi': 1, 'mom': 1, 'how': 1, 'are': 2, 'you': 2, 'we': 1, 'still': 1, 'on': 1, 'for': 1, 'dinner': 1, 'let': 1, 's': 1, 'catch': 1, 'up': 1, 'tomorrow': 2, 'at': 2, 'the': 3, 'office': 2, 'meeting': 2, '3': 1, 'pm': 1, 'team': 1, 'in': 1, 'can': 1, 'send': 1, 'report': 1}

SPAM token counts:
{'free': 2, 'money': 1, 'now': 1, 'lowest': 1, 'price': 1, 'for': 2, 'your': 1, 'meds': 1, 'win': 1, 'a': 1, 'iphone': 1, 'today': 1, 'get': 1, '50': 1, 'off': 1, 'limited': 1, 'time': 1, 'click': 1, 'here': 1, 'prizes': 1}

Total tokens per class:
{'HAM': 34, 'SPAM': 22}


## 3) Compute Class Priors for HAM and SPAM

Compute priors from document counts:

$$
P(\text{HAM}) = \frac{N_{\text{HAM}}}{N}, \quad P(\text{SPAM}) = \frac{N_{\text{SPAM}}}{N}
$$

In [3]:
N = len(train_labels)
class_doc_counts = Counter(train_labels)

priors = {c: class_doc_counts[c] / N for c in ["HAM", "SPAM"]}
log_priors = {c: math.log(priors[c]) for c in ["HAM", "SPAM"]}

print("Document counts:", dict(class_doc_counts))
print("Priors:", priors)
print("Log-priors:", log_priors)

Document counts: {'SPAM': 5, 'HAM': 6}
Priors: {'HAM': 0.5454545454545454, 'SPAM': 0.45454545454545453}
Log-priors: {'HAM': -0.6061358035703156, 'SPAM': -0.7884573603642702}


## 4) Compute Token Likelihoods per Class (Laplace Smoothing)

Use:

$$
P(w\mid c) = \frac{\text{count}(w,c)+1}{\sum_{w'}\text{count}(w',c)+|V|}
$$

Store likelihoods and inspect selected tokens used in test sentences.

In [4]:
likelihoods = {"HAM": {}, "SPAM": {}}
log_likelihoods = {"HAM": {}, "SPAM": {}}

for c in ["HAM", "SPAM"]:
    denom = class_total_tokens[c] + V
    for w in vocab:
        prob = (class_token_counts[c][w] + 1) / denom
        likelihoods[c][w] = prob
        log_likelihoods[c][w] = math.log(prob)

key_tokens = ["limited", "offer", "click", "here", "meeting", "pm", "manager"]
print("Likelihood table for selected tokens:")
print("token\tP(token|HAM)\tP(token|SPAM)")
for token in key_tokens:
    ham_p = likelihoods["HAM"].get(token, 1 / (class_total_tokens["HAM"] + V))
    spam_p = likelihoods["SPAM"].get(token, 1 / (class_total_tokens["SPAM"] + V))
    print(f"{token}\t{ham_p:.6f}\t{spam_p:.6f}")

Likelihood table for selected tokens:
token	P(token|HAM)	P(token|SPAM)
limited	0.012658	0.029851
offer	0.012658	0.014925
click	0.012658	0.029851
here	0.012658	0.029851
meeting	0.037975	0.014925
pm	0.025316	0.014925
manager	0.012658	0.014925


## 5) Implement Manual Naïve Bayes Prediction (Log-Posterior)

For each class:

$$
\log P(c\mid d) \propto \log P(c) + \sum_{w\in d}\log P(w\mid c)
$$

Unseen tokens use Laplace-smoothed fallback probability with count $=0$.

In [5]:
def manual_predict(text: str):
    tokens = preprocess(text)
    scores = {}

    for c in ["HAM", "SPAM"]:
        score = log_priors[c]
        denom = class_total_tokens[c] + V
        unseen_log_prob = math.log(1 / denom)

        for token in tokens:
            if token in log_likelihoods[c]:
                score += log_likelihoods[c][token]
            else:
                score += unseen_log_prob

        scores[c] = score

    predicted_class = max(scores, key=scores.get)
    return predicted_class, tokens, scores

## 6) Classify Required Test Sentences Manually

For each test sentence, we show:
- **Each token's contribution**: log P(token|HAM) vs log P(token|SPAM)
- **Which class each word favors**: Helps understand why a sentence is classified as SPAM or HAM
- **Cumulative scores**: Running totals showing how evidence accumulates
- **Final prediction**: The class with the higher log-posterior score

In [6]:
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

manual_results = {}

for text in test_sentences:
    pred, tokens, scores = manual_predict(text)
    manual_results[text] = {"prediction": pred, "tokens": tokens, "scores": scores}

    print(f"\n{'='*70}")
    print(f"Sentence: {text}")
    print(f"Tokens: {tokens}")
    print(f"{'='*70}")
    
    # Show detailed breakdown for each token
    print(f"\n{'Token':<15} {'log P(w|HAM)':<15} {'log P(w|SPAM)':<15} {'Favors'}")
    print("-" * 70)
    
    # Start with priors
    print(f"{'<PRIOR>':<15} {log_priors['HAM']:<15.6f} {log_priors['SPAM']:<15.6f}")
    
    ham_cumulative = log_priors['HAM']
    spam_cumulative = log_priors['SPAM']
    
    for token in tokens:
        denom_ham = class_total_tokens['HAM'] + V
        denom_spam = class_total_tokens['SPAM'] + V
        
        if token in log_likelihoods['HAM']:
            log_p_ham = log_likelihoods['HAM'][token]
        else:
            log_p_ham = math.log(1 / denom_ham)
        
        if token in log_likelihoods['SPAM']:
            log_p_spam = log_likelihoods['SPAM'][token]
        else:
            log_p_spam = math.log(1 / denom_spam)
        
        ham_cumulative += log_p_ham
        spam_cumulative += log_p_spam
        
        favors = "SPAM" if log_p_spam > log_p_ham else "HAM"
        print(f"{token:<15} {log_p_ham:<15.6f} {log_p_spam:<15.6f} {favors}")
    
    print("-" * 70)
    print(f"{'TOTAL':<15} {ham_cumulative:<15.6f} {spam_cumulative:<15.6f}")
    print(f"\n>>> PREDICTION: {pred} <<<\n")


Sentence: Limited offer, click here!
Tokens: ['limited', 'offer', 'click', 'here']

Token           log P(w|HAM)    log P(w|SPAM)   Favors
----------------------------------------------------------------------
<PRIOR>         -0.606136       -0.788457      
limited         -4.369448       -3.511545       SPAM
offer           -4.369448       -4.204693       SPAM
click           -4.369448       -3.511545       SPAM
here            -4.369448       -3.511545       SPAM
----------------------------------------------------------------------
TOTAL           -18.083927      -15.527786     

>>> PREDICTION: SPAM <<<


Sentence: Meeting at 2 PM with the manager.
Tokens: ['meeting', 'at', '2', 'pm', 'with', 'the', 'manager']

Token           log P(w|HAM)    log P(w|SPAM)   Favors
----------------------------------------------------------------------
<PRIOR>         -0.606136       -0.788457      
meeting         -3.270836       -4.204693       HAM
at              -3.270836       -4.204693       

## 7) Train Multinomial Naïve Bayes with Scikit-Learn

Use `CountVectorizer` + `MultinomialNB` on the same dataset.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Keep preprocessing aligned with manual implementation
clf = Pipeline([
    (
        "vectorizer",
        CountVectorizer(
            tokenizer=preprocess,
            preprocessor=None,
            token_pattern=None,
            lowercase=False,
        ),
    ),
    ("nb", MultinomialNB(alpha=1.0)),
])

clf.fit(train_docs, train_labels)
print("Scikit-learn model trained successfully.")

Scikit-learn model trained successfully.


## 8) Classify Required Test Sentences with Scikit-Learn

In [8]:
sk_preds = clf.predict(test_sentences)
sk_proba = clf.predict_proba(test_sentences)
classes = list(clf.named_steps["nb"].classes_)

sklearn_results = {}

for text, pred, proba in zip(test_sentences, sk_preds, sk_proba):
    prob_map = {cls: float(p) for cls, p in zip(classes, proba)}
    sklearn_results[text] = {"prediction": pred, "probabilities": prob_map}

    print(f"Sentence: {text}")
    print(f"Scikit-learn prediction: {pred}")
    print(f"Class probabilities: {prob_map}")
    print("-" * 60)

Sentence: Limited offer, click here!
Scikit-learn prediction: SPAM
Class probabilities: {np.str_('HAM'): 0.08383194421591078, np.str_('SPAM'): 0.9161680557840887}
------------------------------------------------------------
Sentence: Meeting at 2 PM with the manager.
Scikit-learn prediction: HAM
Class probabilities: {np.str_('HAM'): 0.9781180172810696, np.str_('SPAM'): 0.021881982718930985}
------------------------------------------------------------


## 9) Compare Manual vs Scikit-Learn Predictions

In [9]:
import pandas as pd

comparison_data = []
for text in test_sentences:
    manual_pred = manual_results[text]["prediction"]
    sklearn_pred = sklearn_results[text]["prediction"]
    match = "✓" if manual_pred == sklearn_pred else "✗"
    comparison_data.append({
        "Test Sentence": text,
        "Manual": manual_pred,
        "Scikit-Learn": sklearn_pred,
        "Match": match
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

                    Test Sentence Manual Scikit-Learn Match
       Limited offer, click here!   SPAM         SPAM     ✓
Meeting at 2 PM with the manager.    HAM          HAM     ✓


### Conclusion

- The notebook demonstrates manual Naïve Bayes computation end-to-end (BoW, priors, likelihoods, log-posteriors).
- It also demonstrates scikit-learn `MultinomialNB` on the same training set.
- The comparison table above shows whether both methods agree for each required test sentence.
- Minor differences can happen because of preprocessing/tokenization and implementation details, but both models follow the same Naïve Bayes idea.